# INSOMNIA · CICIDS2017 multi-prototype experiment
Compare single-centroid and multi-prototype label estimators, with and without abstention, at 50% and 70% selection. An adaptive single-centroid reference is included.

**Dataset:** [Kaggle CICIDS2017 mirror](https://www.kaggle.com/datasets/chethuhn/network-intrusion-dataset). Download the ZIP, upload it to Colab, and set `DATA_PATH` to its location. An extracted folder is also supported. Set `DEMO = True` for a synthetic smoke test.

**Protocol:** Train on Monday and the training portion of Tuesday. Reserve a stratified subset of Tuesday for calibration before streaming begins. Score each capture from Wednesday onward, then update using only pseudo-labels. The Kaggle feature CSVs are evaluated as whole capture blocks. This avoids assuming a timestamp or relying on row order as a time axis.

This is an INSOMNIA-style experiment on a different release and protocol from the paper's corrected dataset and 50,000-flow windows. The results measure predict-before-update performance, with updates assumed available for the next capture. Update runtime is recorded separately.

In [ ]:
from time import perf_counter
from pathlib import Path, PurePosixPath
from zipfile import ZipFile

import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print
from sklearn.cluster import KMeans
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

DEMO = False
DATA_PATH = '/content/archive.zip'  # ZIP or extracted directory
LABEL_COLUMN = 'Label'
BENIGN_LABEL = 'BENIGN'
CALIBRATION_FRACTION = 0.20
SEED = 42
PROTOTYPES = 3
RADIUS_QUANTILE = 0.95
MIN_MARGIN = 0.15
WINDOW = 250  # synthetic mode only
RATES = [0.5, 0.7]
INITIAL_EPOCHS = 15
UPDATE_EPOCHS = 2

### Load the eight captures in schedule order
The loader checks all eight expected filenames, normalizes column headers, and keeps the attack-family names for reporting. It excludes timestamps, flow IDs, and endpoint IP columns when present. Median imputation and scaling are fitted on the initial training rows only.

In [ ]:
def synthetic_flows():
    rng = np.random.default_rng(SEED)
    n = 6500
    family = rng.choice(['BENIGN', 'BruteForce', 'DoS'], size=n, p=[.72, .14, .14])
    shifted = rng.choice(np.arange(4550, n), size=400, replace=False)
    family[shifted] = 'PortScan'
    means = {'BENIGN': (0, 0), 'BruteForce': (-3, 3),
             'DoS': (3, 3), 'PortScan': (2, -2)}
    xy = np.array([means[name] for name in family]) + rng.normal(0, .85, (n, 2))
    frame = pd.DataFrame(xy, columns=['feature_1', 'feature_2'])
    frame[LABEL_COLUMN] = family
    return frame


CAPTURES = (
    'Monday-WorkingHours.pcap_ISCX.csv',
    'Tuesday-WorkingHours.pcap_ISCX.csv',
    'Wednesday-workingHours.pcap_ISCX.csv',
    'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv',
    'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv',
    'Friday-WorkingHours-Morning.pcap_ISCX.csv',
    'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv',
    'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv',
)
EXCLUDE = {'timestamp', 'flow id', 'flowid', 'source ip', 'destination ip',
           'src ip', 'dst ip', 'srcip', 'dstip', LABEL_COLUMN.casefold()}


def read_capture(file):
    frame = pd.read_csv(file, encoding='cp1252', low_memory=False)
    frame.columns = frame.columns.str.strip()
    if LABEL_COLUMN not in frame or frame[LABEL_COLUMN].isna().any():
        raise ValueError('Each capture must contain a complete Label column.')
    families = frame.pop(LABEL_COLUMN).astype(str).str.strip().to_numpy()
    if (families == '').any():
        raise ValueError('Blank labels found in a capture.')
    frame = frame.loc[:, [col.casefold() not in EXCLUDE for col in frame.columns]]
    frame = frame.apply(pd.to_numeric, errors='raise').astype(np.float32)
    return frame, families


def load_captures(path):
    expected = [name.casefold() for name in CAPTURES]

    def ordered(items, basename, read):
        found = {}
        for item in items:
            key = basename(item).casefold()
            if key in expected:
                if key in found:
                    raise ValueError(f'Duplicate capture file: {key}')
                found[key] = item
        missing = [name for name in CAPTURES if name.casefold() not in found]
        if missing:
            raise ValueError(f'Missing capture files: {missing}')
        return [read(found[key]) for key in expected]

    path = Path(path)
    if path.is_dir():
        return ordered(path.rglob('*'), lambda p: p.name, read_capture)
    if not path.is_file():
        raise FileNotFoundError(f'Upload the Kaggle archive and set DATA_PATH: {path}')
    with ZipFile(path) as archive:
        def read_member(member):
            with archive.open(member) as file:
                return read_capture(file)
        return ordered(archive.namelist(), lambda p: PurePosixPath(p).name, read_member)


def binary_labels(families):
    return (np.char.lower(families.astype(str)) != BENIGN_LABEL.casefold()).astype(int)


if DEMO:
    data = synthetic_flows()
    families = data.pop(LABEL_COLUMN).to_numpy()
    initial, calibration = data.iloc[:3575], data.iloc[3575:4550]
    initial_families, calibration_families = families[:3575], families[3575:4550]
    captures = [(data.iloc[i:i+WINDOW], families[i:i+WINDOW])
                for i in range(4550, len(data), WINDOW)]
    capture_names = [f'Synthetic block {i+1}' for i in range(len(captures))]
else:
    captures = load_captures(DATA_PATH)
    monday, monday_families = captures.pop(0)
    tuesday, tuesday_families = captures.pop(0)
    train_rows, calibration_rows = train_test_split(
        np.arange(len(tuesday)), test_size=CALIBRATION_FRACTION,
        stratify=binary_labels(tuesday_families), random_state=SEED,
    )
    train_rows.sort()
    calibration_rows.sort()
    if set(monday.columns) != set(tuesday.columns):
        raise ValueError('Monday and Tuesday have different flow feature columns.')
    initial = pd.concat([monday, tuesday.iloc[train_rows]], ignore_index=True)
    initial_families = np.concatenate([monday_families, tuesday_families[train_rows]])
    calibration = tuesday.iloc[calibration_rows]
    calibration_families = tuesday_families[calibration_rows]
    capture_names = list(CAPTURES[2:])
    del monday, tuesday

y_train = binary_labels(initial_families)
y_cal = binary_labels(calibration_families)
if len(np.unique(y_train)) < 2 or len(np.unique(y_cal)) < 2:
    raise ValueError('Initial training and calibration must each contain benign and attack flows.')
input_columns = set(initial.columns)
initial = initial.replace([np.inf, -np.inf], np.nan)
train_medians = initial.median().dropna()
if train_medians.empty:
    raise ValueError('No usable flow features found.')
initial = initial[train_medians.index].fillna(train_medians)
scaler = StandardScaler().fit(initial)
X_train = scaler.transform(initial).astype(np.float32)
del initial


def transform(frame):
    if set(frame.columns) != input_columns:
        raise ValueError('Capture feature columns differ from the training feature columns.')
    clean = frame[train_medians.index].replace([np.inf, -np.inf], np.nan)
    return scaler.transform(clean.fillna(train_medians)).astype(np.float32)


X_cal = transform(calibration)
X_blocks, family_blocks = [], []
for frame, families in captures:
    X_blocks.append(transform(frame))
    family_blocks.append(families)
stream_family = np.concatenate(family_blocks)
y_stream = binary_labels(stream_family)
del captures, frame, calibration
print(f'Rows: {len(X_train):,} training, {len(X_cal):,} calibration, {len(y_stream):,} streaming')
print(f'Features: {X_train.shape[1]}')
display(pd.DataFrame({'capture': capture_names, 'flows': [len(block) for block in X_blocks]}))


### Estimator and abstention
Each class has its own K-means centers. The estimator compares the nearest benign center with the nearest attack center. Abstention requires both a minimum relative distance margin and an in-class distance within a class-specific radius. Radii are computed using true class labels from the pre-stream calibration period.

In [ ]:
class PrototypeEstimator:
    def __init__(self, n_prototypes, abstain=False):
        self.n_prototypes = n_prototypes
        self.abstain = abstain

    def fit(self, X, y, X_cal, y_cal):
        self.centers = []
        self.counts = np.bincount(y, minlength=2).astype(np.int64)
        for label in (0, 1):
            group = X[y == label]
            k = min(self.n_prototypes, len(group))
            if k == 1:
                centers = group.mean(axis=0, keepdims=True, dtype=np.float64)
            else:
                centers = KMeans(n_clusters=k, n_init=5, random_state=SEED).fit(group).cluster_centers_
            self.centers.append(centers)
        distances = self.distances(X_cal)
        self.radii = np.array([
            np.quantile(distances[y_cal == label, label], RADIUS_QUANTILE)
            for label in (0, 1)
        ])
        return self

    def distances(self, X):
        return np.column_stack([
            np.minimum.reduce([np.linalg.norm(X - center, axis=1) for center in centers])
            for centers in self.centers
        ])

    def estimate(self, X):
        distances = self.distances(X)
        labels = distances.argmin(axis=1)
        nearest = distances.min(axis=1)
        rival = distances.max(axis=1)
        margin = (rival - nearest) / np.maximum(rival, 1e-8)
        accepted = np.ones(len(X), dtype=bool)
        if self.abstain:
            accepted = (margin >= MIN_MARGIN) & (nearest <= self.radii[labels])
        return labels, accepted

    def update_single_centroids(self, X, labels):
        for label in (0, 1):
            group = X[labels == label]
            if len(group):
                count = self.counts[label]
                self.centers[label][0] = (
                    count * self.centers[label][0] + group.sum(axis=0, dtype=np.float64)
                ) / (count + len(group))
                self.counts[label] += len(group)

In [ ]:
def run_experiment(name, prototypes, abstain, rate, adaptive=False):
    estimator = PrototypeEstimator(prototypes, abstain).fit(X_train, y_train, X_cal, y_cal)
    model = MLPClassifier(hidden_layer_sizes=(32, 16), batch_size=256, random_state=SEED)
    for _ in range(INITIAL_EPOCHS):
        model.partial_fit(X_train, y_train, classes=[0, 1])

    X_history, y_history = X_train, y_train
    scores, pseudo, window_rows = [], [], []
    start = 0
    for capture, X_window in zip(capture_names, X_blocks):
        probability = model.predict_proba(X_window)[:, 1]
        scores.append(probability)
        budget = max(1, int(rate * len(X_window)))
        selected = np.argsort(np.abs(probability - 0.5), kind='stable')[:budget]
        t0 = perf_counter()
        labels, accepted = estimator.estimate(X_window[selected])
        kept = selected[accepted]
        kept_labels = labels[accepted]
        pseudo.append(pd.DataFrame({
            'row': start + selected, 'pseudo_label': labels, 'accepted': accepted
        }))
        if len(kept):
            X_history = np.concatenate([X_history, X_window[kept]])
            y_history = np.concatenate([y_history, kept_labels])
            for _ in range(UPDATE_EPOCHS):
                model.partial_fit(X_history, y_history)
            if adaptive:
                estimator.update_single_centroids(X_window[kept], kept_labels)
        window_rows.append({
            'capture': capture, 'start': start, 'end': start + len(X_window), 'selected': budget,
            'accepted': len(kept), 'update_seconds': perf_counter() - t0
        })

        start += len(X_window)

    # Ground truth for the stream is accessed only after all updates are complete.
    scores = np.concatenate(scores)
    prediction = scores >= 0.5
    ledger = pd.concat(pseudo, ignore_index=True)
    accepted = ledger.loc[ledger['accepted']]
    true = y_stream[accepted['row'].to_numpy()]
    estimated = accepted['pseudo_label'].to_numpy()
    summary = {
        'estimator': name, 'selection_rate': rate,
        'attack_F1': f1_score(y_stream, prediction, zero_division=0),
        'attack_precision': precision_score(y_stream, prediction, zero_division=0),
        'attack_recall': recall_score(y_stream, prediction, zero_division=0),
        'attack_AP': average_precision_score(y_stream, scores) if y_stream.any() else np.nan,
        'acceptance_rate': len(accepted) / len(ledger),
        'pseudo_error': np.mean(true != estimated) if len(accepted) else np.nan,
        'attack_pseudo_precision': np.mean(true[estimated == 1] == 1) if (estimated == 1).any() else np.nan,
        'update_seconds': sum(row['update_seconds'] for row in window_rows),
    }
    family_recall = {
        label: prediction[stream_family == label].mean()
        for label in sorted(set(stream_family[y_stream == 1]))
    }
    for row in window_rows:
        section = slice(row['start'], row['end'])
        row['attack_F1'] = f1_score(y_stream[section], prediction[section], zero_division=0)
    return summary, family_recall, window_rows


configurations = [
    ('INSOMNIA-style adaptive single centroid', 1, False, True),
    ('Single centroid', 1, False, False),
    ('Single centroid + abstention', 1, True, False),
    ('Multi-prototype', PROTOTYPES, False, False),
    ('Multi-prototype + abstention', PROTOTYPES, True, False),
]
results, family_results, window_results = [], [], []
for rate in RATES:
    for name, prototypes, abstain, adaptive in configurations:
        summary, family_recall, windows = run_experiment(name, prototypes, abstain, rate, adaptive)
        results.append(summary)
        family_results.append({'estimator': name, 'selection_rate': rate, **family_recall})
        window_results.extend({'estimator': name, 'selection_rate': rate, **w} for w in windows)

display(pd.DataFrame(results).round(4))
display(pd.DataFrame(family_results).round(4))
window_metrics = pd.DataFrame(window_results)

Compare the four controlled runs at each selection rate. Read pseudo-label precision alongside acceptance and per-family recall. A method that accepts few examples can raise precision and still hinder adaptation. The synthetic results are a smoke test only. For a research claim, repeat on chronological benchmark data with multiple seeds and choose configuration using pre-stream data.